In [ ]:
import os
import sys
import subprocess

def run(cmd):
    print(f"[RUN] {' '.join(cmd)}")
    subprocess.run(cmd, check=True)

def install_dependencies():
    run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
    run([sys.executable, "-m", "pip", "cache", "purge"])

                                               
    if "COLAB_GPU" in os.environ and os.environ.get("COLAB_GPU"):
        print("[INFO] Colab GPU detected → встановлюємо CUDA-бінарі")
        index_flag = "--extra-index-url"
        index_url  = "https://download.pytorch.org/whl/cu118"
    else:
        print("[INFO] CPU-only середовище або немає COLAB_GPU → встановлюємо CPU-бінарі")
        index_flag = "--index-url"
        index_url  = "https://download.pytorch.org/whl/cpu"

                                                    
    run([
        sys.executable, "-m", "pip", "install",
        index_flag, index_url,
        "torch", "torchvision", "torchaudio"
    ])

                        
    extras = [
        "pymupdf",                            
        "tqdm",                           
        "sentence-transformers",
        "accelerate",
        "bitsandbytes"
    ]
    run([sys.executable, "-m", "pip", "install", "--upgrade"] + extras)

                                                         
    try:
        import torch
        if torch.cuda.is_available():
            print("[INFO] Встановлюємо flash-attn")
            run([sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"])
        else:
            print("[INFO] Flash-attn пропускається — немає доступної CUDA")
    except Exception as e:
        print(f"[WARN] Не вдалося встановити flash-attn: {e}. Продовжуємо без нього.")

                  
    import torch, torchvision
    print(f"[OK] torch       {torch.__version__}")
    print(f"[OK] torchvision {torchvision.__version__}")
    print(f"[OK] CUDA avail: {torch.cuda.is_available()}")

if __name__ == "__main__":
    install_dependencies()


[RUN] /usr/bin/python3 -m pip uninstall -y torch torchvision torchaudio
[RUN] /usr/bin/python3 -m pip cache purge
[INFO] Colab GPU detected → встановлюємо CUDA-бінарі
[RUN] /usr/bin/python3 -m pip install --extra-index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio
[RUN] /usr/bin/python3 -m pip install --upgrade pymupdf tqdm sentence-transformers accelerate bitsandbytes
[INFO] Встановлюємо flash-attn
[RUN] /usr/bin/python3 -m pip install flash-attn --no-build-isolation
[WARN] Не вдалося встановити flash-attn: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'flash-attn', '--no-build-isolation']' returned non-zero exit status 1.. Продовжуємо без нього.
[OK] torch       2.7.0+cu118
[OK] torchvision 0.22.0+cu118
[OK] CUDA avail: True


In [6]:
import os
import requests

                   
pdf_path = "ragfile.pdf"

                                        
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")
                            
  url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
  filename = pdf_path
  response = requests.get(url)

  if response.status_code == 200:
      with open(filename, "wb") as file:
          file.write(response.content)
      print(f"The file has been downloaded and saved as {filename}")
  else:
      print(f"Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} exists.")

File ragfile.pdf exists.


In [7]:
import fitz          
from tqdm.auto import tqdm                      

def text_formatter(text: str) -> str:
    """Виконує базове форматування тексту."""
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """Відкриває PDF, читає текст по сторінках та збирає статистику."""
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc), total=len(doc)):
        text = page.get_text()                          
        text = text_formatter(text)
        pages_and_texts.append({
            "page_number": page_number - 9, # Коригування номера сторінки (на випадок, якщо PDF нумерація починається пізніше)
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),                             
            "page_token_count": len(text) / 4, # Приблизна кількість токенів (1 токен ~ 4 символи)
            "text": text
        })
    doc.close()
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)

                               
print(pages_and_texts[:2])

                                   
import pandas as pd
df = pd.DataFrame(pages_and_texts)
print(df.head())
print(df.describe().round(2))

  0%|          | 0/1221 [00:00<?, ?it/s]

[{'page_number': -9, 'page_char_count': 0, 'page_word_count': 1, 'page_sentence_count_raw': 1, 'page_token_count': 0.0, 'text': ''}, {'page_number': -8, 'page_char_count': 0, 'page_word_count': 1, 'page_sentence_count_raw': 1, 'page_token_count': 0.0, 'text': ''}]
   page_number  page_char_count  page_word_count  page_sentence_count_raw  \
0           -9                0                1                        1   
1           -8                0                1                        1   
2           -7              234               52                        3   
3           -6             1854              325                        8   
4           -5             2081              347                       17   

   page_token_count                                               text  
0              0.00                                                     
1              0.00                                                     
2             58.50  Chemistry 2e                    

In [8]:
from spacy.lang.en import English

nlp = English()
                                         
nlp.add_pipe("sentencizer")

                               
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2

                             
list(doc.sents)

[This is a sentence., This another sentence.]

In [9]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
                                   
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
                                                 
    item["page_sentence_count_spacy"] = len(item["sentences"])

                                                
df = pd.DataFrame(pages_and_texts)
print(df.describe().round(2))

  0%|          | 0/1221 [00:00<?, ?it/s]

       page_number  page_char_count  page_word_count  page_sentence_count_raw  \
count      1221.00          1221.00          1221.00                  1221.00   
mean        601.00          2028.52           333.00                    15.22   
std         352.62           978.31           160.78                    10.26   
min          -9.00             0.00             1.00                     1.00   
25%         296.00          1356.00           225.00                     8.00   
50%         601.00          1965.00           322.00                    14.00   
75%         906.00          2711.00           444.00                    20.00   
max        1211.00          4876.00           852.00                    57.00   

       page_token_count  page_sentence_count_spacy  
count           1221.00                    1221.00  
mean             507.13                      16.10  
std              244.58                      11.52  
min                0.00                       0.00  
25%  

In [10]:
                                 
num_sentence_chunk_size = 10

def split_list(input_list: list, slice_size: int) -> list[list[str]]:
    """Рекурсивно ділить список на підсписки заданого розміру."""
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

                                             
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

                      
import random
print(random.sample(pages_and_texts, k=1))

                           
df = pd.DataFrame(pages_and_texts)
print(df.describe().round(2))

  0%|          | 0/1221 [00:00<?, ?it/s]

[{'page_number': 844, 'page_char_count': 1969, 'page_word_count': 316, 'page_sentence_count_raw': 14, 'page_token_count': 492.25, 'text': '17.6 Corrosion LEARNING OBJECTIVES By the end of this section, you will be able to: • Define corrosion • List some of the methods used to prevent or slow corrosion Corrosion is usually defined as the degradation of metals by a naturally occurring electrochemical process. The formation of rust on iron, tarnish on silver, and the blue-green patina that develops on copper are all examples of corrosion. The total cost of corrosion remediation in the United States is significant, with estimates in excess of half a trillion dollars a year. Chemistry in Everyday Life Statue of Liberty: Changing Colors The Statue of Liberty is a landmark every American recognizes. The Statue of Liberty is easily identified by its height, stance, and unique blue-green color (Figure 17.15). When this statue was first delivered from France, its appearance was not green. It was

In [11]:
import re

                                                   
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
                                               
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
                                                                                      
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

                              
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split(" "))
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # Приблизна кількість токенів

        pages_and_chunks.append(chunk_dict)

print(f"Загальна кількість чанків: {len(pages_and_chunks)}")

                            
print(random.sample(pages_and_chunks, k=1))

                       
df = pd.DataFrame(pages_and_chunks)
print(df.describe().round(2))

  0%|          | 0/1221 [00:00<?, ?it/s]

Загальна кількість чанків: 2555
[{'page_number': 35, 'sentence_chunk': 'Thus, the units of density are defined by the base units of mass and length. The density of a substance is the ratio of the mass of a sample of the substance to its volume. The SI unit for density is the kilogram per cubic meter (kg/m3). For many situations, however, this is an inconvenient unit, and we often use grams per cubic centimeter (g/cm3) for the densities of solids and liquids, and grams per liter (g/L) for gases. Although there are exceptions, most liquids and solids have densities that range from about 0.7 g/cm3 (the density of gasoline) to 19 g/cm3 (the density of gold). The density of air is about 1.2 g/L. Table 1.4 1.4 • Measurements 31', 'chunk_char_count': 659, 'chunk_word_count': 120, 'chunk_token_count': 164.75}]
       page_number  chunk_char_count  chunk_word_count  chunk_token_count
count      2555.00           2555.00           2555.00            2555.00
mean        605.40            967.73  

In [12]:
min_token_length = 30
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
print(f"Кількість чанків після фільтрації: {len(pages_and_chunks_over_min_token_len)}")

Кількість чанків після фільтрації: 2441


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

                                           
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

                                
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device=device)

                                                
single_sentence = "Це приклад речення для ембедингу."
single_embedding = embedding_model.encode(single_sentence)
print(f"Речення: {single_sentence}")
print(f"Розмірність ембедингу: {single_embedding.shape}")

                                                                            
                                     
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

                             
text_chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size=32,                                        
    convert_to_tensor=True,                            
    show_progress_bar=True                                
)

print(f"Розмірність тензора ембедингів: {text_chunk_embeddings.shape}")

                                  
for i, item in enumerate(pages_and_chunks_over_min_token_len):
    item["embedding"] = text_chunk_embeddings[i].cpu().numpy()                                   


Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Речення: Це приклад речення для ембедингу.
Розмірність ембедингу: (768,)


Batches:   0%|          | 0/77 [00:00<?, ?it/s]

Розмірність тензора ембедингів: torch.Size([2441, 768])


In [14]:
                       
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

                        
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
print(text_chunks_and_embedding_df_load.head())

   page_number                                     sentence_chunk  \
0           -7  Chemistry 2e          SENIOR CONTRIBUTING AUTH...   
1           -6  OpenStax Rice University 6100 Main Street MS-3...   
2           -6  HARDCOVER BOOK ISBN-13 978-1-947172-62-3 B&W P...   
3           -5  OPENSTAX  OpenStax provides free, peer-reviewe...   
4           -4  Study where you want, what you want, when you ...   

   chunk_char_count  chunk_word_count  chunk_token_count  \
0               220                38              55.00   
1              1651               253             412.75   
2               155                25              38.75   
3              2025               291             506.25   
4               348                59              87.00   

                                           embedding  
0  [ 3.06962766e-02 -8.90973769e-03 -3.10062207e-...  
1  [ 7.48645980e-03 -7.65670016e-02 -1.59503687e-...  
2  [ 1.89511254e-02 -3.74696627e-02 -5.33240847e-...  
3  [

In [15]:
import numpy as np

                           
if not os.path.exists(embeddings_df_save_path):
    print(f"Error: File not found at {embeddings_df_save_path}. Please run the embedding creation step first.")
else:
                       
    text_chunks_and_embedding_df = pd.read_csv(embeddings_df_save_path)
                                                         
    text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
                                              
    pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")
                                                                                   
    embeddings = torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()), dtype=torch.float32).to(device)
    print(f"Завантажено {len(pages_and_chunks)} чанків.")
    print(f"Розмір тензора ембедингів: {embeddings.shape}")

Завантажено 2441 чанків.
Розмір тензора ембедингів: torch.Size([2441, 768])


In [16]:
from sentence_transformers import util
from time import perf_counter as timer
import textwrap

def print_wrapped(text, wrap_length=80):
    """Друкує текст з перенесенням рядків."""
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

def retrieve_relevant_resources(query: str,
                                embeddings: torch.tensor,
                                model: SentenceTransformer=embedding_model,
                                n_resources_to_return: int=5,
                                print_time: bool=True,
                                device: str = device):
    """Створює ембединг запиту та повертає топ-k оцінок подібності та індексів."""
                               
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

                                  
    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] Час обчислення подібності для {len(embeddings)} ембедингів: {end_time-start_time:.5f} секунд.")

    scores, indices = torch.topk(input=dot_scores, k=n_resources_to_return)
    return scores, indices

def print_top_results_and_scores(query: str,
                                 embeddings: torch.tensor,
                                 pages_and_chunks: list[dict]=pages_and_chunks,
                                 n_resources_to_return: int=5):
    """Знаходить релевантні ресурси та друкує їх."""
    scores, indices = retrieve_relevant_resources(query=query,
                                                  embeddings=embeddings,
                                                  n_resources_to_return=n_resources_to_return)

    print(f"Запит: {query}\n")
    print("Результати:")
    for score, index in zip(scores, indices):
        print(f"Оцінка подібності: {score:.4f}")
        print("Текст:")
        print_wrapped(pages_and_chunks[index]["sentence_chunk"])
        print(f"Номер сторінки: {pages_and_chunks[index]['page_number']}")
        print("\n")

                
query = "macronutrients functions"
print_top_results_and_scores(query=query, embeddings=embeddings)

               
query = "symptoms of pellagra"
print_top_results_and_scores(query=query, embeddings=embeddings, n_resources_to_return=3)

[INFO] Час обчислення подібності для 2441 ембедингів: 0.01903 секунд.
Запит: macronutrients functions

Результати:
Оцінка подібності: 0.4702
Текст:
Proteins provide about 4 Calories per gram, carbohydrates also provide about 4
Calories per gram, and fats and oils provide about 9 Calories/ g. Nutritional
labels on food packages show the caloric content of one serving of the food, as
well as the breakdown into Calories from each of the three macronutrients
(Figure 5.18).2 Francis D. Reardon et al. “The Snellen human calorimeter
revisited, re-engineered and upgraded: Design and performance
characteristics.”Medical and Biological Engineering and Computing 8
(2006)721–28, http://link.springer.com/article/10.1007/ s11517-006-0086-5.5.2 •
Calorimetry 231
Номер сторінки: 235


Оцінка подібності: 0.4572
Текст:
Carbohydrates can store energy, such as the polysaccharides glycogen in animals
or starch in plants. They also provide structural support, such as the
polysaccharide cellulose in plants a

In [17]:
                                 
try:
    gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
    gpu_memory_gb = round(gpu_memory_bytes / (2**30))
    print(f"Доступно відеопам'яті: {gpu_memory_gb} GB")
except Exception as e:
    print(f"Не вдалося отримати інформацію про GPU: {e}")
    gpu_memory_gb = 0                                            

                                                            
                                            
use_quantization_config = False
model_id = "google/gemma-2b-it"                          

if gpu_memory_gb == 0:
     print("GPU не знайдено або недостатньо пам'яті. Робота LLM буде дуже повільною на CPU.")
                                                                         
elif gpu_memory_gb < 5.1:
    print(f"Пам'ять GPU ({gpu_memory_gb}GB) може бути недостатньою для Gemma без квантизації.")
    print("Спробуємо Gemma 2B з 4-бітною квантизацією.")
    use_quantization_config = True
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 8.1:
    print(f"Пам'ять GPU: {gpu_memory_gb}GB | Рекомендована модель: Gemma 2B (4-bit)")
    use_quantization_config = True
    model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
    print(f"Пам'ять GPU: {gpu_memory_gb}GB | Рекомендована модель: Gemma 2B (float16) або Gemma 7B (4-bit)")
                                                                       
    if gpu_memory_gb >= 8.1:                         
         print("Вибираємо Gemma 7B (4-bit).")
         use_quantization_config = True
         model_id = "google/gemma-7b-it"
    else:
         print("Вибираємо Gemma 2B (float16).")
         use_quantization_config = False
         model_id = "google/gemma-2b-it"
elif gpu_memory_gb >= 19.0:
    print(f"Пам'ять GPU: {gpu_memory_gb}GB | Рекомендована модель: Gemma 7B (4-bit або float16)")
    print("Вибираємо Gemma 7B (float16).")                 
    use_quantization_config = False
    model_id = "google/gemma-7b-it"

print(f"Обрано model_id: {model_id}")
print(f"Використовувати квантизацію (4-bit): {use_quantization_config}")

Доступно відеопам'яті: 15 GB
Пам'ять GPU: 15GB | Рекомендована модель: Gemma 2B (float16) або Gemma 7B (4-bit)
Вибираємо Gemma 7B (4-bit).
Обрано model_id: google/gemma-7b-it
Використовувати квантизацію (4-bit): True


In [19]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers.utils import is_flash_attn_2_available

                                                                       
try:
    login()
except Exception as e:
    print(f"Помилка автентифікації Hugging Face Hub: {e}")
    print("Переконайтеся, що ви увійшли через `huggingface-cli login` або налаштували токен.")

                                                       
quantization_config = None
if use_quantization_config:
    quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                             bnb_4bit_compute_dtype=torch.float16)
    print("[INFO] Створено конфігурацію 4-бітної квантизації.")

                                                                            
attn_implementation = "sdpa"                                                     
if device == "cuda" and (is_flash_attn_2_available()) and (torch.cuda.get_device_capability(0)[0] >= 8):
  print("[INFO] Flash Attention 2 доступний, вмикаємо.")
  attn_implementation = "flash_attention_2"
else:
  print(f"[INFO] Flash Attention 2 недоступний або не підтримується GPU. Використовується: {attn_implementation}")

                              
print(f"[INFO] Завантаження токенізатора для {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)
print("[INFO] Токенізатор завантажено.")

                            
print(f"[INFO] Завантаження моделі {model_id} (це може зайняти час)...")
llm_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id,
    torch_dtype=torch.float16,                                              
    quantization_config=quantization_config,                                             
    low_cpu_mem_usage=False,                                                    
    attn_implementation=attn_implementation,                                        
    device_map="auto"                                                                 
)
print("[INFO] Модель LLM завантажено.")

                                                                                   
                                                      
                          
                                                                                              

                                                           
def get_model_num_params(model: torch.nn.Module):
    return sum(param.numel() for param in model.parameters())

def get_model_mem_size(model: torch.nn.Module):
    mem_params = sum(param.nelement() * param.element_size() for param in model.parameters())
    mem_buffers = sum(buf.nelement() * buf.element_size() for buf in model.buffers())
    model_mem_bytes = mem_params + mem_buffers
    model_mem_gb = model_mem_bytes / (1024**3)
    return {"model_mem_gb": round(model_mem_gb, 2)}

print(f"[INFO] Кількість параметрів моделі: {get_model_num_params(llm_model)}")
print(f"[INFO] Розмір моделі в пам'яті: {get_model_mem_size(llm_model)['model_mem_gb']} GB")


[INFO] Створено конфігурацію 4-бітної квантизації.
[INFO] Flash Attention 2 недоступний або не підтримується GPU. Використовується: sdpa
[INFO] Завантаження токенізатора для google/gemma-7b-it...


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

[INFO] Модель LLM завантажено.
[INFO] Кількість параметрів моделі: 4662144000
[INFO] Розмір моделі в пам'яті: 5.07 GB


In [20]:
input_text = "What is the difference between molecular and ionic compounds?"
print(f"Вхідний текст:\n{input_text}")

                                     
dialogue_template = [
    {"role": "user", "content": input_text}
]

                          
prompt = tokenizer.apply_chat_template(
    conversation=dialogue_template,
    tokenize=False,                        
    add_generation_prompt=True                                         
)
print(f"\nПромпт (форматований):\n{prompt}")

                                 
input_ids = tokenizer(prompt, return_tensors="pt").to(device)                                 

                     
print("\n[INFO] Генерація відповіді LLM (без RAG)...")
outputs = llm_model.generate(
    **input_ids,
    max_new_tokens=256                                      
)
print("[INFO] Генерація завершена.")

                          
output_text = tokenizer.decode(outputs[0])
print(f"\nВідповідь моделі (декодована):\n{output_text}")

                                                       
cleaned_output = output_text.replace(prompt, "").replace("<bos>", "").replace("<eos>", "").strip()
print(f"\nВідповідь моделі (очищена):\n{cleaned_output}")

Вхідний текст:
What is the difference between molecular and ionic compounds?

Промпт (форматований):
<bos><start_of_turn>user
What is the difference between molecular and ionic compounds?<end_of_turn>
<start_of_turn>model


[INFO] Генерація відповіді LLM (без RAG)...
[INFO] Генерація завершена.

Відповідь моделі (декодована):
<bos><bos><start_of_turn>user
What is the difference between molecular and ionic compounds?<end_of_turn>
<start_of_turn>model
Sure, here is the difference between molecular and ionic compounds:

**Molecular Compounds:**

* Formed by molecules, which are neutral electrically charged groups of atoms that are held together by covalent bonds.
* Molecules are neutral overall, but the atoms that make up the molecule can have positive or negative electrical charges.
* Examples include carbon dioxide, hydrogen gas, and water.

**Ionic Compounds:**

* Formed by ions, which are atoms that have gained or lost electrons to achieve a stable electron configuration.
* Ions are e

In [23]:
def prompt_formatter(query: str, context_items: list[dict]) -> str:
    """Формує промпт для LLM, доповнюючи запит контекстом з предметної області 'Хімія'."""
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])

    base_prompt = """На основі наведених нижче контекстних елементів, будь ласка, дайте відповідь на запит користувача.
Спочатку виділіть релевантні уривки з контексту, а потім сформулюйте відповідь.
Повертайте лише остаточну відповідь, без етапу виділення уривків.
Намагайтеся, щоб відповіді були максимально пояснювальними та науково точними.
Використовуйте наступні приклади як зразок ідеального стилю відповіді.

Приклад 1:
Запит: Що таке ковалентний зв'язок?
Відповідь: Ковалентний зв'язок виникає в результаті спільного використання електронних пар атомами. Він характерний для неметалів і забезпечує стабільність молекули завдяки заповненню зовнішніх електронних оболонок атомів. Наприклад, у молекулі води (H₂O) кожен атом водню утворює ковалентний зв'язок з атомом кисню.

Приклад 2:
Запит: Яка різниця між ендотермічними та екзотермічними реакціями?
Відповідь: Ендотермічні реакції поглинають енергію у вигляді тепла з навколишнього середовища, тоді як екзотермічні реакції виділяють тепло. Наприклад, процес фотосинтезу є ендотермічним, тоді як горіння деревини — екзотермічним. Різниця визначається зміною ентальпії (ΔH): для ендотермічних реакцій ΔH позитивне, для екзотермічних — негативне.

Приклад 3:
Запит: Що таке pH-розчин і як його виміряти?
Відповідь: pH розчину — це міра концентрації йонів водню (H⁺) у розчині. Чим нижчий pH, тим вищою є кислотність. Значення pH можна виміряти за допомогою pH-метра або індикаторних папірців. Наприклад, чиста вода має нейтральний pH ≈ 7, тоді як оцтова кислота має pH близько 3, що свідчить про її кислотні властивості.

Тепер використайте наступні контекстні елементи, щоб відповісти на запит користувача:
{context}

Запит користувача: {query}
Відповідь:"""

    base_prompt = base_prompt.format(context=context, query=query)

    dialogue_template = [
        {"role": "user", "content": base_prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        conversation=dialogue_template,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt

                                 
test_query = "What are electrolytes and what role do they play in aqueous solutions?"
test_scores, test_indices = retrieve_relevant_resources(query=test_query, embeddings=embeddings, n_resources_to_return=3)
test_context_items = [pages_and_chunks[i] for i in test_indices]
test_prompt = prompt_formatter(query=test_query, context_items=test_context_items)
print(f"Тестовий RAG промпт:\n{test_prompt}")


[INFO] Час обчислення подібності для 2441 ембедингів: 0.00008 секунд.
Тестовий RAG промпт:
<bos><start_of_turn>user
На основі наведених нижче контекстних елементів, будь ласка, дайте відповідь на запит користувача.
Спочатку виділіть релевантні уривки з контексту, а потім сформулюйте відповідь.
Повертайте лише остаточну відповідь, без етапу виділення уривків.
Намагайтеся, щоб відповіді були максимально пояснювальними та науково точними.
Використовуйте наступні приклади як зразок ідеального стилю відповіді.

Приклад 1:
Запит: Що таке ковалентний зв'язок?
Відповідь: Ковалентний зв'язок виникає в результаті спільного використання електронних пар атомами. Він характерний для неметалів і забезпечує стабільність молекули завдяки заповненню зовнішніх електронних оболонок атомів. Наприклад, у молекулі води (H₂O) кожен атом водню утворює ковалентний зв'язок з атомом кисню.

Приклад 2:
Запит: Яка різниця між ендотермічними та екзотермічними реакціями?
Відповідь: Ендотермічні реакції поглинають ен

In [31]:
def ask(query: str,
        temperature: float = 0.7,
        max_new_tokens: int = 256,
        format_answer_text: bool = True,
        return_answer_only: bool = True) -> str | tuple[str, list[dict]]:
    """
    Виконує повний RAG цикл: пошук, доповнення, генерація.
    """
    # 1. Пошук (Retrieval)
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        n_resources_to_return=10                                   
    )

                                            
    context_items = [pages_and_chunks[i] for i in indices]
                                                        
    for i, item in enumerate(context_items):
        item["score"] = scores[i].cpu().item()

    # 2. Доповнення (Augmentation)
    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

                        
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)

    # 3. Генерація (Generation)
    outputs = llm_model.generate(
        **input_ids,
        temperature=temperature,
        do_sample=True,
        max_new_tokens=max_new_tokens
    )

                         
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

                                   
    if format_answer_text:
        output_text = output_text.replace(prompt, "").replace("<bos>", "").replace("<eos>", "")
        output_text = output_text.strip()

                          
    if return_answer_only:
        return output_text
    else:
        return output_text, context_items

                                                                 

final_query = "What is the difference between ionic and covalent bonds?"
print(f"Запит: {final_query}")

                        
rag_answer, rag_context = ask(query=final_query, temperature=0.7, max_new_tokens=512, return_answer_only=False)

print("\nВідповідь RAG:")
print_wrapped(rag_answer)

print("\nВикористаний контекст (перші 2 елементи):")
for item in rag_context[:2]:
    print(f"Оцінка подібності: {item['score']:.4f}")
    print("Текст:")
    print_wrapped(item['sentence_chunk'])
    print(f"Номер сторінки: {item['page_number']}")
    print("-" * 10)

                 
final_query_2 = "How does a catalyst affect a chemical reaction?"
print(f"\nЗапит: {final_query_2}")

rag_answer_2 = ask(query=final_query_2, temperature=0.2, max_new_tokens=256, return_answer_only=True)
print("\nВідповідь RAG:")
print_wrapped(rag_answer_2)


Запит: What is the difference between ionic and covalent bonds?
[INFO] Час обчислення подібності для 2441 ембедингів: 0.00007 секунд.

Відповідь RAG:
**Ionic and Covalent Bonds**  Ionic bonds are formed when atoms transfer
electrons from one atom to another atom, resulting in the formation of
positively and negatively charged ions. Covalent bonds are formed when atoms
share electrons with each other to achieve a stable electron configuration.
**Key Differences Between Ionic and Covalent Bonds:**  * **Nature:** Ionic bonds
are formed between positively and negatively charged ions, while covalent bonds
are formed between atoms that share electrons. * **Polarity:** Ionic bonds are
polar, while covalent bonds can be nonpolar or polar. * **Strength:** Ionic
bonds are typically strong, while covalent bonds can vary in strength. *
**Physical Properties:** Ionic compounds are typically solids at room
temperature, while covalent compounds can be solids, liquids, or gases. *
**Dissolving:** Ioni

In [29]:
final_query_3 = "Describe the structure of the atom, including the roles of protons, neutrons, and electrons, and explain how isotopes differ from each other."
print(f"\nЗапит: {final_query_3}")
rag_answer_3 = ask(query=final_query_3, temperature=1.0, max_new_tokens=256, return_answer_only=True)                                
print("\nВідповідь RAG:")
print_wrapped(rag_answer_3)


Запит: Describe the structure of the atom, including the roles of protons, neutrons, and electrons, and explain how isotopes differ from each other.
[INFO] Час обчислення подібності для 2441 ембедингів: 0.00010 секунд.

Відповідь RAG:
**Атомна структура та ізотони**  Атом, це основна частинка, яка складається з
ядра і електронних оболонок. Ядро містить протони (позитивно заряджені частинки)
і нейтрони (нейтральні частинки). Електронні оболонки містять електрони
(негативно заряджені частинки).  Ядро є надзвичайно щілним, що дозволило
зберегти протони та нейтрони на своїх місцях. Ядро має радіус, який є приблизно
в 10 000 разів меншим радіусом атома.  Ізотони — це атоми одного і того ж
хімічного елемента, які мають однакову кількість протонів, але різну кількість
нейтронів. Ізотони відрізняються своїми masami, але мають однакову хімію.
Наприклад, вуглець має різні ізотони, які відрізняються за масою: вуглець-12,
вуглець-13, вуглець-14 і вуглець-16. Вони мають однакову кількість
